# Supersingular factory — extend the SS bijections to p ≤ 8192

**Just hit Run All** (designed for a long unattended run on the second laptop; the store is
checkpointed every 20 primes and re-running skips primes already present, so the run is
resumable).

For every supersingular prime p in the range this computes the signature ↔ lattice
bijection from scratch (`ecqf_full_bijection_ss`: rigid l-set chosen by minimum eigenline
degree, curve side via one-direction Vélu cycle walks), runs the **validation battery** on
the fresh entry (signature/form sets exact, genuine supersingularity, Φ_ℓ edge check at an
unused split prime, root conventions), and stores it — an entry that fails any check is
reported and left out, never written.

Estimated full run for the 856 primes in (1024, 8192]: **~1.4 h**.
Requires a checkout at commit `1aa838e` or later (qf_isogs l+1 fix, SS pin/sib descriptor
support, walk builder, validation battery) — pull before running.

In [ ]:
import os
os.chdir('../../pycode')
from ss_bij_cache import load, populate, validate_entry
from nt import primesBetween

data = load()
have = sorted(int(p) for p in data)
todo = [p for p in primesBetween(5, 8192) if str(p) not in data]
print(f'store: {len(have)} primes (up to {max(have)}); {len(todo)} to compute in (5, 8192]')

In [ ]:
# the main run: compute + validate + store every missing prime (resumable)
data = populate(pmin=5, pmax=8192, validate=True)

In [ ]:
# post-run: coverage + an independent spot re-validation of STORED entries
import random
data = load()
have = sorted(int(p) for p in data)
missing = [p for p in primesBetween(5, 8192) if str(p) not in data]
print(f'{len(have)} primes stored; missing: {missing if missing else "none"}')
random.seed(0)
for p in random.sample(have, 10):
    res = validate_entry(p, data[str(p)])
    flags = ' '.join(f'{k}={v}' for k, v in res['checks'].items())
    print(f'p={p:>4}: ok={res["ok"]}  {flags}')

When the run finishes, commit **both** data files together (they are kept in sync):
`pycode/data/ecqf_ss_pcbij_velu_4_1024.json` and `pycode/data/ssfp_pc_bij_velu.json`.

Notes
-----
* The file name still says `4_1024`; the loaders (`ecqf_tools.ecqf_ss_1K_pc`,
  `ecfp.ss_precomputed_dictionary`) point at it, so renaming to a range-neutral name is
  deferred until those loaders are touched.
* Once this data lands, the phi factory can draw supersingular Φ_ℓ rows at every fresh
  prime covered here (~√ℓ extra curves per class).